In [3]:
from google.colab import drive; drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!git clone -q https://github.com/ddkang/advex-uar.git
!git clone -q https://github.com/uiuc-focal-lab/RAMP.git
!cd advex-uar && pip install -q --no-deps .

fatal: destination path 'advex-uar' already exists and is not an empty directory.
fatal: destination path 'RAMP' already exists and is not an empty directory.
  Preparing metadata (setup.py) ... done


In [5]:
import functools, json, sys, torch, torch.nn as nn
from torchvision import datasets, transforms
from advex_uar.attacks import PGDAttack, FrankWolfeAttack, JPEGAttack, ElasticAttack
sys.path.insert(0, '/content/RAMP')
from model_zoo.fast_models import PreActResNet18

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
N_EX = 500 # first 500 CIFAR-10 test images

In [6]:
def make_attack(name, eps, iters, step):
    if name == 'pgd_linf': return functools.partial(PGDAttack, iters, eps, step, 32, norm='linf')
    if name == 'pgd_l2':   return functools.partial(PGDAttack, iters, eps, step, 32, norm='l2')
    if name == 'fw_l1':    return functools.partial(FrankWolfeAttack, iters, eps, 32)
    if name.startswith('jpeg'):
        return functools.partial(JPEGAttack, iters, eps, step, 32, opt=name.split('_')[1])
    if name == 'elastic':
        return functools.partial(ElasticAttack, iters, eps, step, 32,
                                 kernel_size=5, kernel_std=3.0/224.0*32)

class Wrap(nn.Module):
    """advex-uar feeds ImageNet-normalized input. RAMP models want [0,1]. Undo it."""
    def __init__(self, m):
        super().__init__(); self.m = m
        self.mu = torch.tensor(MEAN).view(1,3,1,1).cuda()
        self.sd = torch.tensor(STD).view(1,3,1,1).cuda()
    def forward(self, x): return self.m(x * self.sd + self.mu)

def load(path):
    m = PreActResNet18(10, activation='softplus1').cuda()
    sd = torch.load(path, map_location='cuda')
    m.load_state_dict(sd['state_dict'] if 'state_dict' in sd else sd)
    return Wrap(m).cuda().eval()

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
ds = torch.utils.data.Subset(datasets.CIFAR10('./data', train=False, download=True, transform=tf),
                             range(N_EX))
loader = torch.utils.data.DataLoader(ds, batch_size=100, num_workers=2)

100%|██████████| 170M/170M [24:57<00:00, 114kB/s]


In [7]:
def run_one(model, attack_fn):
    """Targeted attack at a random wrong class, scored against the true label."""
    torch.manual_seed(0)
    atk = attack_fn(); std = adv = n = 0
    for x, y in loader:
        x, y = x.cuda(), y.cuda()
        with torch.no_grad():
            std += (model(x).argmax(1) == y).sum().item()
        r = torch.randint(0, 9, y.size(), device='cuda')
        x_adv = atk(model, x, (y + r + 1) % 10, avoid_target=False, scale_eps=False)
        with torch.no_grad():
            adv += (model(x_adv).argmax(1) == y).sum().item()
        n += len(y)
    return std / n, adv / n

In [8]:
CALIBS = json.load(open('/content/advex-uar/advex_uar/analysis/calibrations/cifar-10/calibs.out'))

def sweep(tag, ckpt):
    model, out = load(ckpt), []
    for (atk, eps, iters, step), ata, _ in CALIBS:
        std, adv = run_one(model, make_attack(atk, eps, iters, step))
        out.append({'attack': atk, 'adv_acc': adv, 'ata': ata})
        print('{:10} {:<10} clean={:.3f} adv={:.3f}'.format(atk, eps, std, adv))
        json.dump(out, open('/content/drive/MyDrive/%s.json' % tag, 'w'))
    return out

res_linf = sweep('linf', '/content/RAMP/models/pretr_Linf.pth')

no input normalization
pgd_linf   1          clean=0.854 adv=0.846
pgd_linf   2          clean=0.854 adv=0.840
pgd_linf   4          clean=0.854 adv=0.814
pgd_linf   8          clean=0.854 adv=0.742
pgd_linf   16         clean=0.854 adv=0.398
pgd_linf   32         clean=0.854 adv=0.022
pgd_l2     40         clean=0.854 adv=0.830
pgd_l2     80         clean=0.854 adv=0.814
pgd_l2     160        clean=0.854 adv=0.746
pgd_l2     320        clean=0.854 adv=0.508
pgd_l2     640        clean=0.854 adv=0.028
pgd_l2     2560       clean=0.854 adv=0.000
fw_l1      0.0634766  clean=0.854 adv=0.824
fw_l1      0.1269531  clean=0.854 adv=0.822
fw_l1      0.2539063  clean=0.854 adv=0.794
fw_l1      0.5078125  clean=0.854 adv=0.700
fw_l1      2.03125    clean=0.854 adv=0.134
fw_l1      8.125      clean=0.854 adv=0.000
jpeg_linf  0.03125    clean=0.854 adv=0.822
jpeg_linf  0.0625     clean=0.854 adv=0.790
jpeg_linf  0.125      clean=0.854 adv=0.662
jpeg_linf  0.25       clean=0.854 adv=0.134
jpeg_linf

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.13/dist-packages/advex_uar/attacks/elastic.py:94: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  return F.grid_sample(im, flow, mode='bilinear')


elastic    0.125      clean=0.854 adv=0.764
elastic    0.25       clean=0.854 adv=0.642
elastic    0.5        clean=0.854 adv=0.222
elastic    1          clean=0.854 adv=0.004
elastic    2          clean=0.854 adv=0.000
elastic    8          clean=0.854 adv=0.002


In [11]:
res_l2 = sweep('l2', '/content/RAMP/models/pretr_L2.pth')

no input normalization
pgd_linf   1          clean=0.888 adv=0.880
pgd_linf   2          clean=0.888 adv=0.876
pgd_linf   4          clean=0.888 adv=0.816
pgd_linf   8          clean=0.888 adv=0.630
pgd_linf   16         clean=0.888 adv=0.078
pgd_linf   32         clean=0.888 adv=0.000
pgd_l2     40         clean=0.888 adv=0.872
pgd_l2     80         clean=0.888 adv=0.862
pgd_l2     160        clean=0.888 adv=0.808
pgd_l2     320        clean=0.888 adv=0.590
pgd_l2     640        clean=0.888 adv=0.030
pgd_l2     2560       clean=0.888 adv=0.000
fw_l1      0.0634766  clean=0.888 adv=0.882
fw_l1      0.1269531  clean=0.888 adv=0.878
fw_l1      0.2539063  clean=0.888 adv=0.864
fw_l1      0.5078125  clean=0.888 adv=0.832
fw_l1      2.03125    clean=0.888 adv=0.338
fw_l1      8.125      clean=0.888 adv=0.000
jpeg_linf  0.03125    clean=0.888 adv=0.876
jpeg_linf  0.0625     clean=0.888 adv=0.868
jpeg_linf  0.125      clean=0.888 adv=0.754
jpeg_linf  0.25       clean=0.888 adv=0.258
jpeg_linf

In [22]:
!pip install -q tqdm git+https://github.com/RobustBench/robustbench.git

!cd /content/RAMP && python RAMP.py \
    --data_dir /content/data \
    --save_dir /content/drive/MyDrive \
    --fname ramp_hw \
    --model_name pretr_Linf \
    --finetune_model \
    --lr-max 0.05 \
    --lr-schedule piecewise-ft \
    --at_iter 10 \
    --epochs 3 \
    --kl --max --seed 0

  Preparing metadata (setup.py) ... done
/usr/local/lib/python3.13/dist-packages/robustbench/loaders.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
ramp_hw
data loaded on cuda:0
no input normalization
pretrained model loaded
[train] Linf eps=0.03137 iters=10, L1 eps=12.00000 iters=10, L2 eps=0.50000 iters=10
[eval] Linf eps=0.03137, L2 eps=0.50000, L1 eps=12.00000
100% 391/391 [13:16<00:00,  2.04s/batch, loss=1.37, acc_s=0.5, acc_t=0.52]
[epoch] 1 [time] 796.1 s [train] loss 1.37359
100% 391/391 [13:17<00:00,  2.04s/batch, loss=1.16, acc_s=0.574, acc_t=0.598]
[epoch] 2 [time] 797.0 s [train] loss 1.16306
100% 391/391 [13:18<00:00,  2.04s/batch, loss=1.11, acc_s=0.594, acc_t=0.619]
[epoch] 3 [time] 798.8 s [train] loss 1.10868[eval train] Linf 62.5% L2 79.0% L1 67.0%

In [23]:
res_ramp = sweep('ramp', '/content/drive/MyDrive/ramp_hw/ep_3_0.pth')

no input normalization
pgd_linf   1          clean=0.824 adv=0.820
pgd_linf   2          clean=0.824 adv=0.814
pgd_linf   4          clean=0.824 adv=0.810
pgd_linf   8          clean=0.824 adv=0.714
pgd_linf   16         clean=0.824 adv=0.394
pgd_linf   32         clean=0.824 adv=0.018
pgd_l2     40         clean=0.824 adv=0.820
pgd_l2     80         clean=0.824 adv=0.814
pgd_l2     160        clean=0.824 adv=0.782
pgd_l2     320        clean=0.824 adv=0.672
pgd_l2     640        clean=0.824 adv=0.242
pgd_l2     2560       clean=0.824 adv=0.000
fw_l1      0.0634766  clean=0.824 adv=0.824
fw_l1      0.1269531  clean=0.824 adv=0.814
fw_l1      0.2539063  clean=0.824 adv=0.804
fw_l1      0.5078125  clean=0.824 adv=0.778
fw_l1      2.03125    clean=0.824 adv=0.590
fw_l1      8.125      clean=0.824 adv=0.000
jpeg_linf  0.03125    clean=0.824 adv=0.816
jpeg_linf  0.0625     clean=0.824 adv=0.808
jpeg_linf  0.125      clean=0.824 adv=0.760
jpeg_linf  0.25       clean=0.824 adv=0.564
jpeg_linf

/usr/local/lib/python3.13/dist-packages/advex_uar/attacks/elastic.py:94: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  return F.grid_sample(im, flow, mode='bilinear')


elastic    0.125      clean=0.824 adv=0.744
elastic    0.25       clean=0.824 adv=0.642
elastic    0.5        clean=0.824 adv=0.286
elastic    1          clean=0.824 adv=0.030
elastic    2          clean=0.824 adv=0.002
elastic    8          clean=0.824 adv=0.002


In [24]:
def uar(results):
  """UAR per attack: sum of adversarial accuracy over the 6 calibrated epsilons, divided by the sum of the published ATA values (Kang et al., advex-uar README)."""
  scores = {}
  for attack in set(r['attack'] for r in results):
      rows = [r for r in results if r['attack'] == attack]
      scores[attack] = (
          sum(r['adv_acc'] for r in rows) /
          sum(r['ata'] for r in rows)
      )
  return scores

In [25]:
for name, results in [('Linf-AT', res_linf),
                    ('L2-AT', res_l2),
                    ('RAMP', res_ramp)]:
  scores = uar(results)
  unseen = [scores[a] for a in
            ('jpeg_linf', 'jpeg_l2', 'jpeg_l1', 'elastic')]
  print(name,
        'unseen UAR =', round(sum(unseen) / 4, 3),
        'all 7 UAR =', round(sum(scores.values()) / 7, 3))

Linf-AT unseen UAR = 0.517 all 7 UAR = 0.644
L2-AT unseen UAR = 0.552 all 7 UAR = 0.676
RAMP unseen UAR = 0.616 all 7 UAR = 0.73
